# M3L2 E03 - ChatOpenAI: el modelo como objeto

## Un solo concepto

Este notebook se enfoca en **un unico componente**: `ChatOpenAI`.

La lecture (Seccion 9) dice:

> "LangChain envuelve modelos de lenguaje en interfaces estandar.
> Si cambias de modelo, idealmente cambias solo la configuracion."

## Que vamos a ver

1. La diferencia entre llamar `openai` directamente y usar `ChatOpenAI`.
2. Que tipo devuelve `llm.invoke()` (spoiler: NO es un string).
3. Como configurar el modelo en un solo lugar.

## Necesita API key de OpenAI


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


## Por que necesitamos un wrapper del modelo (Lecture M3L2 - Seccion 9)

La lecture M3L2 dice:

> "LangChain envuelve modelos de lenguaje en interfaces estandar.
> El resto del pipeline no necesita conocer todos los detalles de la API del proveedor.
> Si cambias de modelo, idealmente cambias solo la configuracion."
> -- Lecture M3L2, Seccion 9.2

**Conexion con M3L1**: en M3L1, el modelo estaba hardcodeado dentro del agente:

```python
# M3L1: el modelo vive dentro del agente, mezclado con la logica
def agente_manual(consulta, ciudad):
    ...
    response = client.chat.completions.create(
        model="gpt-4o-mini",   # <- hardcodeado dentro de la funcion
        messages=[...],
        temperature=0,
    )
```

**Con LangChain** (M3L2), el modelo es un objeto separado:

```text
M3L1 (hardcodeado)                M3L2 (wrapper encapsulado)
---------------------------------  ----------------------------------
openai.chat.completions.create(    llm = ChatOpenAI(
    model="gpt-4o-mini",              model="gpt-4o-mini",
    messages=[...],                    temperature=0
    temperature=0,                 )
)                                  # llm se pasa como parametro
                                   # al prompt, chain, agent, etc.
```

### Buenas practicas del LLM wrapper (Lecture M3L2 - Seccion 9.3)

| Practica | Por que importa |
|---|---|
| Configurar el modelo en un solo lugar | Si cambia el modelo, cambias 1 linea |
| Usar temperature=0 para clasificacion | Respuestas reproducibles |
| Documentar el modelo usado | El equipo sabe que version se usa |
| Medir costo y latencia | Produccion requiere monitoreo |
| No hardcodear en muchos archivos | Evita el problema de M3L1 |


## Sin LangChain: llamada directa

Con `openai` directamente:
- el modelo esta hardcodeado en la llamada,
- el formato de los mensajes es especifico de OpenAI,
- la respuesta viene en `response.choices[0].message.content`.


In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Di solo la palabra: hola"}],
    temperature=0,
)

texto = response.choices[0].message.content
print(f"Tipo de response: {type(response).__name__}")
print(f"Tipo de texto: {type(texto).__name__}")
print(f"Texto: {texto}")
print()
print("Para extraer el texto: response.choices[0].message.content")
print("Es verboso y especifico de la API de OpenAI")


## Por que ChatOpenAI devuelve AIMessage y no un string

`llm.invoke()` devuelve un objeto `AIMessage` porque LangChain necesita
mantener metadatos junto con el texto:

```text
AIMessage
  |-- content: str         <- el texto de la respuesta
  |-- response_metadata    <- tokens usados, model name, etc.
  |-- id                   <- identificador unico del mensaje
  |-- usage_metadata       <- tokens de entrada y salida
```

Esto es util para tracing y monitoreo (saber cuanto costó cada llamada).

Para obtener solo el texto: `.content` o usar `StrOutputParser` (E04).

**Analogia con M3L1**: en M3L1 escribiamos:
```python
answer = response.choices[0].message.content  # extraer texto a mano
```

LangChain hace ese mismo trabajo, pero con una interfaz estandar
que funciona igual para OpenAI, Anthropic, Gemini, etc.


## Con LangChain: ChatOpenAI

`ChatOpenAI` es un **wrapper**: encapsula el modelo y expone una interfaz estandar.

La interfaz estandar de LangChain es `.invoke(input)`.

Pero ojo: el resultado de `llm.invoke()` es un `AIMessage`, no un string.
Para obtener el texto hay que acceder a `.content`.

(En E04 veremos como `StrOutputParser` soluciona esto automaticamente.)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# TODO 1: crear el objeto ChatOpenAI con model="gpt-4o-mini" y temperature=0
llm = None  # reemplazar

print(f"Tipo: {type(llm).__name__ if llm else 'TODO no completado'}")


In [ ]:
# TODO 2: invocar el modelo con llm.invoke("Di solo la palabra: hola")
# Guarda el resultado en 'result' y observa su tipo
# result = llm.invoke("Di solo la palabra: hola")

# Descomentar despues de completar TODO 1 y 2:
# print(f"Tipo de result: {type(result).__name__}")
# print(f"result: {result}")
# print()
# print(f"result.content: {result.content}")
# print()
# print("ChatOpenAI devuelve un AIMessage, no un string.")
# print("Para obtener el texto: result.content")


## El modelo es configurable

Como `llm` es un objeto, puedo crear distintas configuraciones
y pasarlas a diferentes partes del codigo.


In [ ]:
# TODO 3: crear dos configuraciones del mismo modelo con temperaturas distintas
# llm_preciso = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# llm_creativo = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

# Invocar ambos con la misma pregunta y comparar
# pregunta = "Describe el cielo en 5 palabras"
# r_preciso = llm_preciso.invoke(pregunta).content
# r_creativo = llm_creativo.invoke(pregunta).content
# print(f"Temperatura 0:   {r_preciso}")
# print(f"Temperatura 1.0: {r_creativo}")


In [ ]:
def run_checks():
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import AIMessage
    assert llm is not None, "TODO 1: llm es None"
    assert isinstance(llm, ChatOpenAI)
    result = llm.invoke("Di solo: test")
    assert isinstance(result, AIMessage), f"llm.invoke devuelve {type(result)}, no AIMessage"
    assert len(result.content) > 0
    print("M3L2 E03 checks passed")

run_checks()


## Cierre

| Llamada directa `openai` | `ChatOpenAI` de LangChain |
|---|---|
| `response.choices[0].message.content` | `result.content` (mas limpio) |
| Modelo hardcodeado en cada llamada | Modelo en un objeto configurable |
| Formato especifico de OpenAI | Interfaz estandar `.invoke()` |
| No se puede pasar como componente | Se pasa como objeto a una chain |

**Siguiente**: E04 muestra como `StrOutputParser` extrae el `.content` automaticamente.
